In [37]:
import pandas as pd

from get_market_cap_by_ticker import get_market_cap_by_ticker
from DATA.stock_invest_function import *
from get_fs_data_by_ticker import extract_quarterly_fs_data
from get_hscode_processed_data import get_hscode_processed_data
from get_monthly_export_data import extract_monthly_exog_var
from get_revenue_export_joined_table import get_revenue_export_joined_table
from sarima_endog_forecast import forecast_endog_with_optional_exog
from sarima_endog_forecast import forecast_endog_fill_tail
from get_forecasted_revenue_df import build_forecast_df_from_out
from revenue_forecast_all_package import *
from get_revenue_ttm_df import get_revenue_ttm_df
from get_psr_from_mc_and_rev import build_psr_series
from psr_forecast_runner import forecast_psr_all_models

# 1) DB 접속정보
db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

ticker = "A140860"
hs_code = '9012101090'

df_mc = get_market_cap_by_ticker(db_info, ticker)

df_rev = extract_quarterly_fs_data(
    db_info=db_info,
    table_name="korea_fs_data",          # 실제 테이블명으로 교체
    target_indicator="매출액(천원)",       # 원하는 지표명
    ticker= ticker,                     # 원하는 종목코드
)

# df_exog['monthly_raw']   # 월별 데이터
# df_exog['quarterly']     # 분기 데이터
# df_exog['exog']

df_export = get_hscode_processed_data(db_info, hs_code = hs_code)
df_exog = df_export['quarterly']
df_exog_monthly =extract_monthly_exog_var(df_export)
combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
    df_rev=df_rev,
    df_exog=df_exog,
    join_how="outer",
    fill_exog="ffill"   # 필요시
)

# combined_df, final_combined_data, forecast_df = get_revenue_export_joined_table(
#     df_rev=df_rev,
#     df_exog=df_exog,
#     join_how="right",
#     fill_exog="ffill"   # 필요시
# )

# combined_df: Date 인덱스 + ['endog_var','exog_var'] (exog_var는 없어도 됨)
# 예: horizon=4 (분기 4개 또는 월 4개)
final_combined_data = final_combined_data.loc['2015-01-01':]

out = forecast_endog_with_optional_exog(
    combined_df=final_combined_data,
    horizon = 4,          # ⬅️ 예측 기간 설정
    hs_code= None,   # ⬅️ None이면 exog_var 사용 안함
    # seasonal_period=4 # 직접 지정도 가능(미지정시 자동 추론)
)
rev_forecast_with_noexog = out['forecast']

out2 = forecast_endog_fill_tail(final_combined_data, hs_code= hs_code)
rev_forecast_with_exog = out2['forecast']

rev_sarima_noexog = build_forecast_df_from_out(out, combined_df=final_combined_data)
rev_sarima_exog = build_forecast_df_from_out(out2, combined_df=final_combined_data)

rev_ets_df     = forecast_revenue_ets(final_combined_data, horizon=4)
rev_prophet_df = forecast_revenue_prophet(final_combined_data, horizon=4)   # Prophet 미설치면 에러
rev_lstm_df    = forecast_revenue_lstm(final_combined_data, horizon=4, lookback=12)
rev_theta_df   = forecast_revenue_theta(final_combined_data, horizon=4)

rev_final = get_revenue_ttm_df(
    df_rev=df_rev,
    rev_sarima_noexog=rev_sarima_noexog,
    rev_sarima_exog=rev_sarima_exog,
    rev_ets_df=rev_ets_df,
    rev_prophet_df=rev_prophet_df,
    rev_theta_df=rev_theta_df,
    rev_lstm_df=rev_lstm_df  # 또는 ref_lstm_df
    # forecast_col_map={"prophet": "yhat"}  # 필요시 예측 컬럼 강제 지정
)

psr_df = build_psr_series(df_mc=df_mc, df_rev=df_rev)

# -------------------------------------------
# psr 예측 파이프라인 (Py3.9 호환)
# -------------------------------------------

# -------- 사용 예시 --------
psr_df = psr_df  # index=DatetimeIndex, column='psr'
exog_df = df_exog_monthly[['exog_var']]
horizon = 13  # 원하는 예측기간
fc_table = forecast_psr_all_models(psr_df, horizon=horizon, exog_df=exog_df)

from valuation_forecast import compute_valuation_forecast

# value_start_date 를 None 으로 두면 “현재 달 + 1개월”이 자동 적용됩니다. (예: 오늘이 2025-10이면 2025-11)
valuation_forecast_result = compute_valuation_forecast(
    fc_table=fc_table,
    rev_final=rev_final,
    value_start_date=None  # 또는 '2025-11' 같이 직접 지정
)

print(valuation_forecast_result.tail())

✅ A140860 시가총액 4,084건 조회 완료
[메모리] forecast_sarima 실행 전: 833.29 MB
[메모리] find_best_sarima_params 실행 전: 833.29 MB
[메모리] find_best_sarima_params 실행 후: 833.30 MB (변화: +0.02 MB)
[메모리] forecast_sarima 실행 후: 833.30 MB (변화: +0.02 MB)
[메모리] find_best_sarima_params 실행 전: 833.30 MB
[메모리] find_best_sarima_params 실행 후: 833.31 MB (변화: +0.00 MB)
[메모리] forecast_ets 실행 전: 833.31 MB


13:05:47 - cmdstanpy - INFO - Chain [1] start processing


[메모리] forecast_ets 실행 후: 833.06 MB (변화: -0.25 MB)
[메모리] forecast_prophet 실행 전: 833.06 MB


13:05:48 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_prophet 실행 후: 833.08 MB (변화: +0.02 MB)
[메모리] forecast_lstm 실행 전: 833.08 MB
[메모리] forecast_lstm 실행 후: 852.40 MB (변화: +19.32 MB)
[메모리] forecast_theta 실행 전: 852.40 MB
[메모리] forecast_theta 실행 후: 852.40 MB (변화: +0.00 MB)
[메모리] forecast_sarima 실행 전: 852.40 MB
[메모리] find_best_sarima_params 실행 전: 852.40 MB
[메모리] find_best_sarima_params 실행 후: 861.95 MB (변화: +9.55 MB)
[메모리] forecast_sarima 실행 후: 853.00 MB (변화: +0.60 MB)
[메모리] forecast_ets 실행 전: 853.00 MB


13:06:06 - cmdstanpy - INFO - Chain [1] start processing
13:06:06 - cmdstanpy - INFO - Chain [1] done processing


[메모리] forecast_ets 실행 후: 853.00 MB (변화: +0.00 MB)
[메모리] forecast_prophet 실행 전: 853.00 MB
[메모리] forecast_prophet 실행 후: 855.26 MB (변화: +2.26 MB)
[메모리] forecast_lstm 실행 전: 855.26 MB
[메모리] forecast_lstm 실행 후: 871.29 MB (변화: +16.02 MB)
[메모리] forecast_theta 실행 전: 871.29 MB
[메모리] forecast_theta 실행 후: 871.29 MB (변화: +0.00 MB)
[경고] SARIMA exog 예측 실패: exog contains inf or nans
         date  mc_sarima_noexog  mc_sarima_exog        mc_ets    mc_prophet  \
63 2026-07-31      2.710758e+09             NaN  2.822197e+09  2.437063e+09   
64 2026-08-31      2.588496e+09             NaN  2.673275e+09  2.284975e+09   
65 2026-09-30               NaN             NaN           NaN           NaN   
66 2026-10-31               NaN             NaN           NaN           NaN   
67 2026-11-30               NaN             NaN           NaN           NaN   

         mc_lstm      mc_theta  
63  3.086959e+09  2.547553e+09  
64  3.097628e+09  2.552828e+09  
65           NaN           NaN  
66           NaN       

In [38]:
fc_table

,psr_SARIMA_noexog,psr_SARIMA_exog,psr_ETS,psr_Prophet,psr_LSTM,psr_Theta
date,,,,,,
2025-11-30,11.940089,NaN,12.347882,13.721077,11.132706,10.377288
2025-12-31,13.536828,NaN,13.525824,15.078044,11.089456,10.399136
2026-01-31,13.518926,NaN,13.847283,15.330820,11.038124,10.420984
2026-02-28,10.356529,NaN,10.408421,11.050580,11.111979,10.442832
2026-03-31,10.277911,NaN,10.583698,11.317922,11.081308,10.464680
2026-04-30,10.576279,NaN,10.645646,11.315414,11.071132,10.486527
2026-05-31,11.216320,NaN,11.477246,12.334471,11.043220,10.508375
2026-06-30,11.927433,NaN,12.033159,12.917013,11.041312,10.530223
2026-07-31,11.700608,NaN,11.932098,12.745060,11.086143,10.552071


In [32]:
exog_df

,exog_var
Date,
2009-01-31,169848.0
2009-02-28,1073510.0
2009-03-31,355161.0
2009-04-30,518808.0
2009-05-31,486568.0
...,...
2026-05-31,6017640.0
2026-06-30,11766300.0
2026-07-31,8342400.0


In [13]:
fc_table

,psr_SARIMA_noexog,psr_SARIMA_exog,psr_ETS,psr_Prophet,psr_LSTM,psr_Theta
date,,,,,,
2025-11-30,11.685938,NaN,12.097617,13.802116,10.867579,10.105045
2025-12-31,13.305707,NaN,13.275534,15.152953,10.844142,10.126833
2026-01-31,13.269604,NaN,13.596945,15.374756,10.834056,10.148622
2026-02-28,10.121583,NaN,10.158081,11.080006,10.860882,10.170410
2026-03-31,10.031721,NaN,10.333377,11.281194,10.789677,10.192199
2026-04-30,10.338636,NaN,10.395375,11.228896,10.741301,10.213987
2026-05-31,10.972006,NaN,11.226929,12.223765,10.701059,10.235776
2026-06-30,11.688318,NaN,11.782847,12.806033,10.701769,10.257564
2026-07-31,11.457447,NaN,11.681766,12.663095,10.755435,10.279353


In [14]:
valuation_forecast_result

,date,mc_sarima_noexog,mc_sarima_exog,mc_ets,mc_prophet,mc_lstm,mc_theta
55,2025-11-30,2.547052e+09,NaN,2.665514e+09,2.934204e+09,2.540520e+09,2.217052e+09
56,2025-12-31,2.968963e+09,NaN,2.990192e+09,2.977495e+09,2.624803e+09,2.484256e+09
57,2026-01-31,2.960907e+09,NaN,3.062587e+09,3.021079e+09,2.622362e+09,2.489602e+09
58,2026-02-28,2.258475e+09,NaN,2.288014e+09,2.177177e+09,2.628855e+09,2.494947e+09
59,2026-03-31,2.253324e+09,NaN,2.358973e+09,2.188231e+09,2.856138e+09,2.430584e+09
60,2026-04-30,2.322264e+09,NaN,2.373126e+09,2.178087e+09,2.843332e+09,2.435780e+09
61,2026-05-31,2.464531e+09,NaN,2.562959e+09,2.371063e+09,2.832680e+09,2.440976e+09
62,2026-06-30,2.707911e+09,NaN,2.786896e+09,2.448722e+09,3.172958e+09,2.476452e+09
63,2026-07-31,2.654423e+09,NaN,2.762988e+09,2.421390e+09,3.188869e+09,2.481712e+09
64,2026-08-31,2.532889e+09,NaN,2.614067e+09,2.278124e+09,3.201266e+09,2.486972e+09


In [13]:
final_combined_data.tail(12)

,endog_var,exog_var
Date,,
2023-12-31,45469063.39,18754960.0
2024-03-31,25664386.43,14738160.0
2024-06-30,44696800.46,22569750.0
2024-09-30,41374479.73,15353560.0
2024-12-31,63324182.80,24615880.0
2025-03-31,50909652.66,16134020.0
2025-06-30,52309189.06,17207080.0
2025-09-30,NaN,17476360.0
2025-12-31,NaN,31035350.0


In [23]:
df_export['monthly_raw'][['expDlr_forecast_12m']].rename(columns={

{'monthly_raw':             expDlr_forecast_12m  year  quarter  month year_quarter  \
 Date                                                                 
 2009-01-31             169848.0  2009        1      1       2009Q1   
 2009-02-28            1073510.0  2009        1      2       2009Q1   
 2009-03-31             355161.0  2009        1      3       2009Q1   
 2009-04-30             518808.0  2009        2      4       2009Q2   
 2009-05-31             486568.0  2009        2      5       2009Q2   
 ...                         ...   ...      ...    ...          ...   
 2026-05-31            6017640.0  2026        2      5       2026Q2   
 2026-06-30           11766300.0  2026        2      6       2026Q2   
 2026-07-31            8342400.0  2026        3      7       2026Q3   
 2026-08-31            6347270.0  2026        3      8       2026Q3   
 2026-09-30            9951200.0  2026        3      9       2026Q3   
 
            root_hs_code  
 Date                     
 2009-

In [62]:
from upload_valuation_longform import upload_valuation_longform

upload_valuation_longform(
    valuation_forecast_result=valuation_forecast_result,
    fc_table=fc_table,
    rev_final=rev_final,
    psr_df=psr_df,
    ticker=ticker,
    db_info=db_info,
    table_name="Korea_company_valuation_ver2",
    forecast_date=None   # None이면 오늘 날짜로 입력
)

[OK] 1330 rows upserted into Korea_company_valuation_ver2 for ticker=A005930 (forecast_date=2025-10-29).


### PSR 측정

### DB에 결과 업로드

In [9]:
final_combined_data

,endog_var,exog_var
Date,,
2004-12-31,0.0,NaN
2005-12-31,0.0,NaN
2006-12-31,0.0,NaN
2007-12-31,0.0,NaN
2008-12-31,0.0,NaN
...,...,...
2025-09-30,NaN,17476360.0
2025-12-31,NaN,31035350.0
2026-03-31,NaN,16796660.0
